# Fake Chat Models

The `fake_chat_models.py` module provides deterministic chat-model implementations for tests.

These models can return fixed responses, cycle through configured response lists, stream characters or message fragments, inject streaming failures, preserve sequential batch ordering, emit token callbacks, and echo the final input message.

# FakeMessagesListChatModel

`FakeMessagesListChatModel` cycles through a configured list of complete `BaseMessage` responses.

After returning the final configured response, the model resets its internal index and begins again from the first response.

## Bases

- `BaseChatModel`

## Attributes

1. `responses`: Stores the complete messages returned by successive model invocations.

   Responses are used in order and then repeated from the beginning.

   * **Type:**
     ```python
     responses: list[BaseMessage]
     ```

2. `sleep`: Stores an optional blocking delay in seconds before each response is returned.
   * **Type:**
     ```python
     sleep: float | None = None
     ```

3. `i`: Stores the index of the response returned by the next invocation.
   * **Type:**
     ```python
     i: int = 0
     ```

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "fake-messages-list-chat-model"
     ```

### Methods

1. `_generate`: Returns the next configured message inside a `ChatResult`.

   When `sleep` is set, execution blocks for that duration before selecting the response. The internal index advances after each invocation and wraps to zero after the final response.

   The input messages, stop sequences, run manager, and additional keyword arguments do not affect response selection.

   * **Syntax:**
     ```python
     _generate(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional generation parameters
     ) -> ChatResult
     ```

# FakeListChatModelError

`FakeListChatModelError` is raised by `FakeListChatModel` when an injected streaming failure reaches the configured chunk index.

## Bases

- `Exception`

# FakeListChatModel

`FakeListChatModel` cycles through a configured list of string responses.

Normal invocation returns one complete string. Streaming yields one character per `ChatGenerationChunk`, and the model can deliberately raise `FakeListChatModelError` at a selected character index.

Its batch methods execute inputs sequentially rather than concurrently so response order remains deterministic.

## Bases

- `SimpleChatModel`

## Attributes

1. `responses`: Stores the string responses returned in cyclic order.
   * **Type:**
     ```python
     responses: list[str]
     ```

2. `sleep`: Stores an optional delay in seconds.

   Normal invocation uses `time.sleep`. Synchronous streaming sleeps before each character, while asynchronous streaming uses `asyncio.sleep`.

   * **Type:**
     ```python
     sleep: float | None = None
     ```

3. `i`: Stores the index of the response used by the next invocation or stream.
   * **Type:**
     ```python
     i: int = 0
     ```

4. `error_on_chunk_number`: Stores the zero-based character index at which streaming raises `FakeListChatModelError`.

   A value of `None` disables injected streaming failures.

   * **Type:**
     ```python
     error_on_chunk_number: int | None = None
     ```

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "fake-list-chat-model"
     ```

2. `_identifying_params`: Returns the configured response list as the model's identifying parameters.
   * **Type:**
     ```python
     _identifying_params: dict[
         str,
         Any
     ]
     ```

   * **Value:**
     ```python
     {
         "responses": self.responses,
     }
     ```

### Methods

1. `_call`: Returns the next configured string response.

   When `sleep` is set, execution blocks before the response is selected. The response index advances and wraps to the beginning after the last response.

   * **Syntax:**
     ```python
     _call(
         self,
         *args: Any, # Positional invocation arguments
         **kwargs: Any # Keyword invocation arguments
     ) -> str
     ```

2. `_stream`: Streams the next configured response one character at a time.

   Every character is wrapped in an `AIMessageChunk` and then in a `ChatGenerationChunk`. The final character receives `chunk_position="last"`.

   When `error_on_chunk_number` matches the current zero-based character index, `FakeListChatModelError` is raised before that character is yielded.

   * **Syntax:**
     ```python
     _stream(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional streaming parameters
     ) -> Iterator[
         ChatGenerationChunk
     ]
     ```

3. `_astream`: Asynchronously streams the next configured response one character at a time.

   It follows the same response cycling, chunk-position, and injected-error rules as `_stream`. When `sleep` is configured, `asyncio.sleep` is awaited before each character.

   * **Syntax:**
     ```python
     async _astream(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Optional async callback manager
         **kwargs: Any # Additional streaming parameters
     ) -> AsyncIterator[
         ChatGenerationChunk
     ]
     ```

4. `batch`: Invokes the model synchronously for each input in list order.

   Inputs are processed sequentially to preserve deterministic response ordering. When `config` is a list, each input is paired with the corresponding configuration through `zip`; unmatched trailing values are ignored.

   The `return_exceptions` parameter is accepted for Runnable compatibility but is not applied by this override, so exceptions propagate normally.

   * **Syntax:**
     ```python
     batch(
         self,
         inputs: list[Any], # Inputs to invoke in order
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input configuration
         *,
         return_exceptions: bool = False, # Accepted compatibility flag
         **kwargs: Any # Additional invocation parameters
     ) -> list[AIMessage]
     ```

5. `abatch`: Invokes the model asynchronously for each input in list order.

   Each invocation is awaited before the next begins, avoiding concurrent access to the cyclic response index. Per-input configurations are paired through `zip`, and unmatched trailing values are ignored.

   The `return_exceptions` parameter is accepted but not applied, so exceptions propagate normally.

   * **Syntax:**
     ```python
     async abatch(
         self,
         inputs: list[Any], # Inputs to invoke in order
         config: RunnableConfig
         | list[RunnableConfig]
         | None = None, # Shared or per-input configuration
         *,
         return_exceptions: bool = False, # Accepted compatibility flag
         **kwargs: Any # Additional invocation parameters
     ) -> list[AIMessage]
     ```

# FakeChatModel

`FakeChatModel` is a minimal chat model that always returns `"fake response"`.

It provides both synchronous string generation and a native asynchronous `ChatResult` implementation.

## Bases

- `SimpleChatModel`

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "fake-chat-model"
     ```

2. `_identifying_params`: Returns a fixed identifying dictionary.
   * **Type:**
     ```python
     _identifying_params: dict[
         str,
         Any
     ]
     ```

   * **Value:**
     ```python
     {
         "key": "fake",
     }
     ```

### Methods

1. `_call`: Returns the fixed string `"fake response"`.
   * **Syntax:**
     ```python
     _call(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional generation parameters
     ) -> str
     ```

2. `_agenerate`: Asynchronously returns the fixed response as an `AIMessage` inside a `ChatResult`.

   No external asynchronous operation is performed.

   * **Syntax:**
     ```python
     async _agenerate(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: AsyncCallbackManagerForLLMRun | None = None, # Optional async callback manager
         **kwargs: Any # Additional generation parameters
     ) -> ChatResult
     ```

# GenericFakeChatModel

`GenericFakeChatModel` consumes responses from a supplied iterator.

Each invocation takes the next iterator value. String values are converted to `AIMessage` objects, while existing `AIMessage` values are returned unchanged.

Its synchronous streaming implementation splits text while preserving whitespace and separately streams provider-specific `additional_kwargs`, including function-call fields.

## Bases

- `BaseChatModel`

## Attributes

1. `messages`: Stores the iterator that supplies successive fake responses.

   Each item must be an `AIMessage` or string. Exhausting the iterator causes the normal `StopIteration` behaviour of `next`.

   * **Type:**
     ```python
     messages: Iterator[
         AIMessage | str
     ]
     ```

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "generic-fake-chat-model"
     ```

### Methods

1. `_generate`: Consumes the next configured response and returns it inside a `ChatResult`.

   String responses are wrapped in `AIMessage`. Existing `AIMessage` values are retained.

   * **Syntax:**
     ```python
     _generate(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional generation parameters
     ) -> ChatResult
     ```

2. `_stream`: Converts the next generated message into synchronous generation chunks.

   Text content is split with a whitespace-capturing regular expression, so spaces and other whitespace are preserved as separate chunks. Every text token triggers `run_manager.on_llm_new_token` when a run manager is available.

   The final text chunk receives `chunk_position="last"` only when the message has no `additional_kwargs`.

   Additional keyword arguments are emitted as zero-content chunks. For `"function_call"` data:

   - String subfield values are split around commas while preserving commas as chunks.
   - Non-string subfield values are emitted as one chunk.
   - Token callbacks receive an empty token string.

   A `ValueError` is raised when generation does not produce `ChatResult`, when its first message is not an `AIMessage`, or when non-empty message content is not a string.

   * **Syntax:**
     ```python
     _stream(
         self,
         messages: list[BaseMessage], # Input messages
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional streaming parameters
     ) -> Iterator[
         ChatGenerationChunk
     ]
     ```

# ParrotFakeChatModel

`ParrotFakeChatModel` returns the final input message unchanged.

It is useful for testing message pipelines without generating new content.

## Bases

- `BaseChatModel`

### Properties

1. `_llm_type`: Returns the identifying model type.
   * **Type:**
     ```python
     _llm_type: str
     ```

   * **Value:**
     ```python
     "parrot-fake-chat-model"
     ```

### Methods

1. `_generate`: Returns the final input message inside a `ChatResult`.

   A `ValueError` is raised when the input message list is empty.

   * **Syntax:**
     ```python
     _generate(
         self,
         messages: list[BaseMessage], # Input messages whose final item is returned
         stop: list[str] | None = None, # Optional stop sequences
         run_manager: CallbackManagerForLLMRun | None = None, # Optional callback manager
         **kwargs: Any # Additional generation parameters
     ) -> ChatResult
     ```

## Inherited Behaviour

These fake models inherit the normal `BaseChatModel` or `SimpleChatModel` interfaces, including invocation, asynchronous invocation, tracing, Runnable composition, and any fallback asynchronous execution not overridden in this file.

Inherited methods already documented in the chat-model base module are not repeated here.